# Real Estate Price Model — Simple Comparison

A clean notebook version of the simplified model-testing script.

### Workflow
1. Load the processed dataset
2. Remove flagged and systematic outliers
3. Group rare property types
4. Frequency-encode location
5. Train on log-transformed prices
6. Compare Random Forest and HistGradientBoosting
7. Cross-validate the best model


## 1. Imports and settings

In [29]:
import os
import glob
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42


## 2. Find and load the dataset

This helper searches common project folders for `data.xlsx`.


In [30]:
def find_data_file(filename="data.xlsx"):
    here = os.getcwd()

    candidates = [
        filename,
        os.path.join(here, filename),
        os.path.join(here, "..", filename),
        os.path.join(here, "..", "data", filename),
        os.path.join(here, "..", "data", "processed", filename),
        os.path.join(here, "data", filename),
        os.path.join(here, "data", "processed", filename),
    ]

    for path in candidates:
        if os.path.exists(path):
            return os.path.abspath(path)

    matches = glob.glob(
        os.path.join(here, "..", "..", "**", filename),
        recursive=True,
    )

    if matches:
        return matches[0]

    raise FileNotFoundError(
        f"Couldn't find {filename} — set the path manually."
    )

DATA_PATH = find_data_file()

print("Loading data from:", DATA_PATH)

df = pd.read_excel(DATA_PATH)

print("Raw shape:", df.shape)

df.head()


Loading data from: d:\Edversity AI Course\real-estate-scraping\Real-Estate-Intelligence-System\data\processed\data.xlsx
Raw shape: (3108, 23)


,URL,Type,Purpose,Area,Bedroom,Bath,Added,Price,Location,Location_city,...,Area_sqft,Suspected_outlier,Log_price,Price_per_sqft,Log_area_sqft,Location_median_ppsf,Area_per_bedroom,Bath_bedroom_ratio,Type_target_enc,City_target_enc
0,https://www.zameen.com/Property/askari_askari_...,Apartment,For Sale,10 Marla,3.0,3.0,33 minutes ago,PKR 3 Crore,"Askari 11, Askari",Lahore,...,2722.50,False,17.216708,11019.28375,7.909306,16332.15270,907.50,1.000000,16.741292,17.480455
1,https://www.zameen.com/Property/gulberg_3_gulb...,Other,For Sale,2.4 Kanal,6.0,7.0,1 hour ago,PKR 15.5 Crore,"Gulberg 3 - Block M, Gulberg 3",Lahore,...,13068.00,False,18.858936,11861.03459,9.477922,15794.30670,2178.00,1.166667,17.994220,17.480455
2,https://www.zameen.com/Property/dha_phase_5_pe...,Apartment,For Sale,9.8 Marla,3.0,4.0,3 hours ago,PKR 6.95 Crore,"Penta Square By DHA Lahore, DHA Phase 5",Lahore,...,2668.05,False,18.056837,26048.98709,7.889103,15794.30670,889.35,1.333333,16.741292,17.480455
3,https://www.zameen.com/Property/dha_phase_7_dh...,House,For Sale,1 Kanal,5.0,7.0,3 hours ago,PKR 14.5 Crore,"DHA Phase 7 - Block Y, DHA Phase 7",Lahore,...,5445.00,False,18.792244,26629.93572,8.602453,20202.02020,1089.00,1.400000,17.538551,17.480455
4,https://www.zameen.com/Property/dha_phase_7_dh...,House,For Sale,1 Kanal,5.0,6.0,3 hours ago,PKR 14.5 Crore,"DHA Phase 7 - Block U, DHA Phase 7",Lahore,...,5445.00,False,18.792244,26629.93572,8.602453,17860.42241,1089.00,1.200000,17.538551,17.480455


## 3. Clean the data

We:
- remove rows already marked as suspected outliers
- trim the lowest and highest 1% of `Price_per_sqft` **within each property type**
- group property types with fewer than 30 rows into `Other_rare`
- frequency-encode detailed locations
- create a log-transformed target


In [31]:
# Remove already flagged outliers
df = df[df["Suspected_outlier"] == False].copy()

# Per-property-type trimming using Price_per_sqft
lo = df.groupby("Type")["Price_per_sqft"].transform(
    lambda s: s.quantile(0.01)
)

hi = df.groupby("Type")["Price_per_sqft"].transform(
    lambda s: s.quantile(0.99)
)

df = df[
    (df["Price_per_sqft"] >= lo)
    & (df["Price_per_sqft"] <= hi)
].copy()

print("Shape after outlier trimming:", df.shape)


Shape after outlier trimming: (3026, 23)


In [32]:
# Group rare property types
type_counts = df["Type"].value_counts()
rare_types = type_counts[type_counts < 30].index

df["Type_grouped"] = df["Type"].where(
    ~df["Type"].isin(rare_types),
    "Other_rare"
)

# Frequency-encode detailed location
df["Location_freq"] = df["Location"].map(
    df["Location"].value_counts()
)

# Log-transform price target
df["y_log"] = np.log1p(df["Price_pkr"])

df["Type_grouped"].value_counts()


Type_grouped
House               1662
Other                384
Apartment            368
Room                 171
Plot                 159
Residential Plot     112
Villa                 86
Other_rare            84
Name: count, dtype: int64

## 4. Prepare features and train/test split

In [33]:
NUM = [
    "Area_sqft",
    "Bedroom",
    "Bath",
    "Log_area_sqft",
    "Area_per_bedroom",
    "Bath_bedroom_ratio",
    "Location_freq",
]

CAT = [
    "Type_grouped",
    "Location_city",
]

X = df[NUM + CAT]
y_log = df["y_log"]
y_raw = df["Price_pkr"]

X_train, X_test, ylog_train, ylog_test, yraw_train, yraw_test = train_test_split(
    X,
    y_log,
    y_raw,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


Training rows: 2420
Testing rows: 606


## 5. Preprocessing

In [34]:
preprocess = ColumnTransformer([
    ("num", "passthrough", NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
])


## 6. Train and compare models

Both models use the same preprocessing and log-price target.

The results are reported back in normal PKR as well as log-space R².


In [35]:
models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_depth=6,
        learning_rate=0.05,
        max_iter=400,
        random_state=RANDOM_STATE,
    ),
}


In [36]:
results = []
fitted_models = {}

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocess),
        ("model", model),
    ])

    pipe.fit(X_train, ylog_train)

    pred_log = pipe.predict(X_test)
    pred_raw = np.expm1(pred_log)

    mae = mean_absolute_error(yraw_test, pred_raw)
    rmse = np.sqrt(mean_squared_error(yraw_test, pred_raw))
    r2 = r2_score(yraw_test, pred_raw)
    log_r2 = r2_score(ylog_test, pred_log)

    results.append({
        "Model": name,
        "MAE_PKR": mae,
        "RMSE_PKR": rmse,
        "R2": r2,
        "Log_R2": log_r2,
    })

    fitted_models[name] = pipe

results_df = (
    pd.DataFrame(results)
    .sort_values("Log_R2", ascending=False)
    .reset_index(drop=True)
)

results_df


,Model,MAE_PKR,RMSE_PKR,R2,Log_R2
0,RandomForest,1.696767e+07,4.315817e+07,0.717468,0.785314
1,HistGradientBoosting,1.881084e+07,4.491592e+07,0.693985,0.771222


## 7. Cross-validate the best model

A single train/test split can be lucky.

This uses shuffled 5-fold cross-validation to check whether the winning model performs consistently across different subsets of the dataset.


In [37]:
best_name = results_df.iloc[0]["Model"]
best_pipe = fitted_models[best_name]

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

scores = cross_validate(
    best_pipe,
    X,
    y_log,
    cv=cv,
    scoring="r2",
    n_jobs=-1,
)

cv_mean = scores["test_score"].mean()
cv_std = scores["test_score"].std()

print("Best model:", best_name)
print("5-fold CV Log R² scores:", scores["test_score"])
print(f"Mean Log R²: {cv_mean:.3f}")
print(f"Standard deviation: {cv_std:.3f}")


Best model: RandomForest
5-fold CV Log R² scores: [0.78781668 0.82076109 0.77404756 0.81070584 0.81328491]
Mean Log R²: 0.801
Standard deviation: 0.018


## 8. Final result

Use the table above for the single 80/20 test split and the cross-validation section to judge stability.

For the hackathon, the stronger model should ideally have:
- lower MAE and RMSE
- higher R² / Log R²
- a good cross-validation average
- a reasonably small cross-validation standard deviation


In [38]:
# Temporary prediction for my Askari 11 apartment

area_sqft = 2250
bedrooms = 3
bathrooms = 3

sample_property = pd.DataFrame([{
    "Area_sqft": area_sqft,
    "Bedroom": bedrooms,
    "Bath": bathrooms,
    "Log_area_sqft": np.log1p(area_sqft),
    "Area_per_bedroom": area_sqft / bedrooms,
    "Bath_bedroom_ratio": bathrooms / bedrooms,

    # Approximate frequency for Askari 11 in the dataset
    "Location_freq": 20,

    "Type_grouped": "Apartment",
    "Location_city": "Lahore"
}])

pred_log = best_pipe.predict(sample_property)
pred_price = np.expm1(pred_log)[0]

print(f"Predicted price: PKR {pred_price:,.0f}")
print(f"Predicted price: {pred_price / 10_000_000:.2f} Crore")

Predicted price: PKR 39,545,344
Predicted price: 3.95 Crore
